# **DeFi Micro Simulation**

Key is to obtain more understanding

In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

In [24]:
import os
import sys

import datetime as dt
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import numpy as np

import math
from dataclasses import dataclass
from math import isfinite
from matplotlib.ticker import FuncFormatter

import warnings
warnings.filterwarnings('ignore')

from typing import Tuple, Dict, Any, List
from pathlib import Path
from IPython.display import display

from datetime import datetime, timedelta, time
import time

from tqdm.notebook import tqdm

src_path = os.path.abspath('../src')
sys.path.append(src_path)

# from prepare_data import load_and_process_data
# from exchange_v2 import ExchangeSim, OpportunisticSpreadWidenMM, SWRParams, run_backtest, BTLogger

pd.options.display.float_format = '{:.6f}'.format
pd.set_option('display.max_columns', None)

In [27]:
# NOTE: Uniswap v2 fee is usually 0.003 (0.30%).
# You currently use 0.0003 (0.03%). I'll keep your value as-is.
FEE = 0.0003
F = 1 - FEE

def swap_y_for_x(x_reserve, y_reserve, dy_in, fee_factor=F):
    dy_eff = dy_in * fee_factor
    k = x_reserve * y_reserve
    y_new = y_reserve + dy_eff
    x_new = k / y_new
    dx_out = x_reserve - x_new
    return dx_out

def get_amount_in(amount_out, reserve_in, reserve_out, fee_factor=F):
    """
    Uniswap v2-style "how much input is needed to get amount_out?"
    Here we use floats (no integer rounding / +1).
    amount_out must be < reserve_out
    """
    if amount_out >= reserve_out:
        return float("inf")
    return (reserve_in * amount_out) / ((reserve_out - amount_out) * fee_factor)

@dataclass
class GasModel:
    gas_used: int = 250_000              # typical for multi-hop + callback (rough)
    base_fee_gwei: float = 15.0          # network condition
    priority_fee_gwei: float = 2.0       # your bid for inclusion
    eth_price_usdc: float = 1100.0       # use an estimate or oracle / pool mid

    def cost_usdc(self) -> float:
        gwei_to_eth = 1e-9
        return self.gas_used * (self.base_fee_gwei + self.priority_fee_gwei) * gwei_to_eth * self.eth_price_usdc

def flashswap_arb_profit_realistic(
    borrow_usdc,
    poolA_eth, poolA_usdc,      # Pool A (cheap ETH)
    poolB_eth, poolB_usdc,      # Pool B (expensive ETH)
    fee_factor=F,
    gas: GasModel | None = None,
    p_success: float = 1.0,     # prob your tx succeeds (not reverted / not outbid)
    report_in_usdc: bool = True # convert ETH profit to USDC using eth_price_usdc
):
    """
    Realistic Uniswap v2 flashswap arb (single flashswap + one external swap):

    - Borrow USDC from Pool B via flashswap (amount1Out = borrow_usdc)
    - Swap that USDC to ETH on Pool A
    - Repay Pool B in ETH (amount0In computed by get_amount_in)
    - Profit is leftover ETH

    Net expected profit accounts for gas always being paid; success probability affects expected profit:
      EV = p_success * profit - gas_cost
    (If reverted, you lose gas but profit is 0.)
    """
    # 1) Use borrowed USDC to buy ETH cheaply on Pool A
    eth_bought = swap_y_for_x(poolA_eth, poolA_usdc, borrow_usdc, fee_factor)

    # 2) Compute exact ETH needed to repay Pool B for borrowing borrow_usdc USDC
    # In Pool B, USDC is the "out" token, ETH is the "in" token.
    eth_repay = get_amount_in(
        amount_out=borrow_usdc,
        reserve_in=poolB_eth,
        reserve_out=poolB_usdc,
        fee_factor=fee_factor
    )

    eth_profit = eth_bought - eth_repay

    gas_cost_usdc = gas.cost_usdc() if gas else 0.0

    if report_in_usdc:
        eth_price = gas.eth_price_usdc if gas else (poolB_usdc / poolB_eth)  # fallback
        profit_usdc = eth_profit * eth_price
        ev_usdc = p_success * profit_usdc - gas_cost_usdc
        return {
            "profit_usdc": profit_usdc,
            "ev_usdc": ev_usdc,
            "eth_bought": eth_bought,
            "eth_repay": eth_repay,
            "eth_profit": eth_profit,
            "gas_cost_usdc": gas_cost_usdc,
            "p_success": p_success
        }
    else:
        ev_eth = p_success * eth_profit - (gas_cost_usdc / (gas.eth_price_usdc if gas else 1.0))
        return {
            "profit_eth": eth_profit,
            "ev_eth": ev_eth,
            "eth_bought": eth_bought,
            "eth_repay": eth_repay,
            "gas_cost_usdc": gas_cost_usdc,
            "p_success": p_success
        }

# Example pools (your narrative)
poolA_eth, poolA_usdc = 100.0, 100_000.0  # ~1000
poolB_eth, poolB_usdc = 100.0, 110_000.0  # ~1100

gas = GasModel(gas_used=250_000, base_fee_gwei=15, priority_fee_gwei=2, eth_price_usdc=1100)

for borrow in [100, 300, 500, 700, 1000, 2000, 3000, 5000, 10_000]:
    r = flashswap_arb_profit_realistic(
        borrow, poolA_eth, poolA_usdc, poolB_eth, poolB_usdc,
        gas=gas,
        p_success=0.9  # example: 85% chance you land; 15% you get outbid/revert
    )
    print(
        f"borrow={borrow:>6,.0f} | eth_bought={r['eth_bought']:.6f} | eth_repay={r['eth_repay']:.6f} "
        f"| eth_profit={r['eth_profit']:.6f} | profit_usdc={r['profit_usdc']:.2f} | "
        f"gas={r['gas_cost_usdc']:.2f} | EV={r['ev_usdc']:.2f}"
    )

borrow=   100 | eth_bought=0.099870 | eth_repay=0.091019 | eth_profit=0.008851 | profit_usdc=9.74 | gas=4.68 | EV=4.09
borrow=   300 | eth_bought=0.299013 | eth_repay=0.273555 | eth_profit=0.025458 | profit_usdc=28.00 | gas=4.68 | EV=20.53
borrow=   500 | eth_bought=0.497364 | eth_repay=0.456758 | eth_profit=0.040606 | profit_usdc=44.67 | gas=4.68 | EV=35.52
borrow=   700 | eth_bought=0.694927 | eth_repay=0.640631 | eth_profit=0.054296 | profit_usdc=59.73 | gas=4.68 | EV=49.08
borrow= 1,000 | eth_bought=0.989805 | eth_repay=0.917707 | eth_profit=0.072098 | profit_usdc=79.31 | gas=4.68 | EV=66.70
borrow= 2,000 | eth_bought=1.960208 | eth_repay=1.852408 | eth_profit=0.107800 | profit_usdc=118.58 | gas=4.68 | EV=102.05
borrow= 3,000 | eth_bought=2.911773 | eth_repay=2.804580 | eth_profit=0.107193 | profit_usdc=117.91 | gas=4.68 | EV=101.45
borrow= 5,000 | eth_bought=4.760544 | eth_repay=4.763334 | eth_profit=-0.002790 | profit_usdc=-3.07 | gas=4.68 | EV=-7.44
borrow=10,000 | eth_bought=9.

In [23]:
best = (-1e18, None, None, None)
for borrow in range(1, 10_000):
    p, eth, out = flashswap_arb_profit(borrow, poolA_eth, poolA_usdc, poolB_eth, poolB_usdc)
    if p > best[0]:
        best = (p, borrow, eth, out)

print("best_profit, best_borrow, eth_bought, usdc_out =", best)

# find approximate break-even
breakeven = []
for borrow in range(1, 10_000):
    p, *_ = flashswap_arb_profit(borrow, poolA_eth, poolA_usdc, poolB_eth, poolB_usdc)
    if p >= 0:
        breakeven.append(borrow)

print("profit>=0 range:", (min(breakeven), max(breakeven)))

best_profit, best_borrow, eth_bought, usdc_out = (117.63732918559981, 2426, 2.3678455013176034, 2543.6373291856)
profit>=0 range: (1, 4969)
